# Snow Pole Detection - Training Pipeline

Train **YOLOv9t**, **YOLO11n**, and **RF-DETR** for snow pole detection using the data-centric, two-stage strategy described in the report.

Pseudo-labeling source video mentioned by the user:
- https://youtu.be/zmTC-fp54aQ?si=AoPFY2IDsuQ2eg0P

**What changed in this notebook**
- Added **two-stage training** support:
  - **Stage 1**: pre-train on large pseudo-labeled YouTube data
  - **Stage 2**: fine-tune on high-quality iPhone / Roadpoles data
- Updated YOLO models to **YOLOv9t** and **YOLO11n**
- Added **RF-DETR** training code
- Raised default image size to **1280** for thin-object detection
- Added optional support for **1920px** experiments and stage-wise checkpoint chaining

**Important Notes**
- Dataset sources can be **read-only**
- YOLO labels are standard **.txt** format
- For YOLO training we create symlinks into a local working directory
- RF-DETR can train directly from the prepared YOLO dataset directory through `data.yaml`


## 1. Setup & Installation

In [1]:
import sys
import subprocess

def install_package(package_name, import_name=None):
    if import_name is None:
        import_name = package_name.split('==')[0].split('[')[0]
    try:
        __import__(import_name)
        print(f"✓ {package_name}")
        return True
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', package_name, '--break-system-packages', '-q']
        )
        return True

print("Checking dependencies...\n")
install_package('torch')
install_package('ultralytics')
install_package('rfdetr')
install_package('pyyaml', 'yaml')


Checking dependencies...

✓ torch
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/maryasae/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ ultralytics


True

In [3]:
import os
import yaml
from pathlib import Path
from datetime import datetime
import json
import time
import shutil
from pprint import pprint

import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from ultralytics import YOLO

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
    RFDETR_AVAILABLE = True
except Exception as e:
    RFDETR_AVAILABLE = False
    print(f"⚠️ RF-DETR import failed: {e}")

print(f"✓ All packages loaded")
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available - will use CPU")
print(f"RF-DETR available: {RFDETR_AVAILABLE}")


✓ All packages loaded

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


## 2. Dataset Configuration

Dataset is **read-only**. Labels are in standard YOLO **.txt** format.
We create symlinks to images and labels in a local working directory.

In [4]:
# ============================================================
# CONFIGURATION - Adjust paths as needed
# ============================================================

# Dataset root (READ-ONLY)
# For Cybele lab:
BASE_PATH = Path("/datasets/tdt4265/Poles2025")
# For IDUN cluster, uncomment instead:
# BASE_PATH = Path("/cluster/projects/vc/courses/TDT17/ad/Poles2025")


# Source YouTube video for the pseudo-labeling pipeline
YOUTUBE_SOURCE_URL = "https://youtu.be/zmTC-fp54aQ?si=AoPFY2IDsuQ2eg0P"

# Optional pseudo-labeled YouTube dataset for Stage 1 pre-training
# Expected YOLO structure:
#   YOUTUBE_PSEUDO_PATH/
#       train/images, train/labels
#       val/images,   val/labels
#       test/images   (optional)
#
# Set to None if you only want Stage 2 fine-tuning on the provided datasets.
YOUTUBE_PSEUDO_PATH = None
# Example:
# YOUTUBE_PSEUDO_PATH = Path("/cluster/work/.../snowpole_youtube_pseudo")

# Local working directory (writable)
WORK_DIR = Path("training_data")
WORK_DIR.mkdir(exist_ok=True)

# Output directory for runs
RUNS_DIR = Path("runs")
RUNS_DIR.mkdir(exist_ok=True)

# Dataset structures - labels are .txt files
DATASETS = {
    "roadpoles_v1": {
        "path": BASE_PATH / "roadpoles_v1",
        "splits": {
            "train": {"images": "train/images", "labels": "train/labels"},
            "val": {"images": "valid/images", "labels": "valid/labels"},
            "test": {"images": "test/images", "labels": None},
        },
    },
    "Road_poles_iPhone": {
        "path": BASE_PATH / "Road_poles_iPhone",
        "splits": {
            "train": {"images": "images/Train/train", "labels": "labels/Train/train"},
            "val": {"images": "images/Validation/val", "labels": "labels/Validation/val"},
            "test": {"images": "images/Test/test", "labels": None},
        },
    },
}

if YOUTUBE_PSEUDO_PATH is not None:
    DATASETS["youtube_pseudo"] = {
        "path": Path(YOUTUBE_PSEUDO_PATH),
        "splits": {
            "train": {"images": "train/images", "labels": "train/labels"},
            "val": {"images": "val/images", "labels": "val/labels"},
            "test": {"images": "test/images", "labels": "test/labels"},
        },
    }

# Stage definitions based on the report:
# Stage 1 -> large noisy pseudo-labeled YouTube data
# Stage 2 -> high-quality domain fine-tuning on iPhone / Roadpoles
STAGE_DATASETS = {
    "stage1": ["youtube_pseudo"],                       # optional
    "stage2": ["Road_poles_iPhone", "roadpoles_v1"],   # robust fine-tuning
}

print(f"Dataset path: {BASE_PATH}")
print(f"Dataset exists: {BASE_PATH.exists()}")
print(f"Working directory: {WORK_DIR.resolve()}")
print(f"YouTube source URL: {YOUTUBE_SOURCE_URL}")
print(f"Configured datasets: {list(DATASETS.keys())}")
print("\nStage plan:")
pprint(STAGE_DATASETS)


Dataset path: /datasets/tdt4265/Poles2025
Dataset exists: True
Working directory: training_data
Selected dataset: combined


## 3. Prepare Dataset(s) for Training

Create stage-specific local datasets for YOLO and RF-DETR.

- **Stage 1** uses the optional pseudo-labeled YouTube dataset
- **Stage 2** uses the high-quality provided datasets for fine-tuning

Each prepared dataset contains:
- `train/images`, `train/labels`
- `val/images`, `val/labels`
- `test/images`, `test/labels` (if available)
- `data.yaml`


In [5]:
def prepare_dataset(datasets_config, work_dir, dataset_names):
    """
    Prepare a YOLO-format dataset in a writable local directory:
    - Create symlinks to images
    - Create symlinks to labels (or copy if symlinks fail)
    - Generate data.yaml
    """
    print(f"\n{'='*80}")
    print(f"Preparing dataset in: {work_dir}")
    print(f"Datasets included: {dataset_names}")
    print(f"{'='*80}")

    if work_dir.exists():
        shutil.rmtree(work_dir)
    for split in ["train", "val", "test"]:
        (work_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (work_dir / split / "labels").mkdir(parents=True, exist_ok=True)

    stats = {
        "train": {"images": 0, "labels": 0},
        "val": {"images": 0, "labels": 0},
        "test": {"images": 0, "labels": 0},
    }

    for dataset_name in dataset_names:
        if dataset_name not in datasets_config:
            print(f"⚠️ Skipping unknown dataset: {dataset_name}")
            continue

        dataset_info = datasets_config[dataset_name]
        dataset_path = Path(dataset_info["path"])

        if not dataset_path.exists():
            print(f"⚠️ Dataset not found: {dataset_path}")
            continue

        print(f"\nProcessing {dataset_name}...")

        prefix = f"{dataset_name}_"

        for split, paths in dataset_info["splits"].items():
            images_path = dataset_path / paths["images"]
            labels_path = dataset_path / paths["labels"] if paths["labels"] else None

            if not images_path.exists():
                print(f"  {split}: images path not found -> {images_path}")
                continue

            image_files = []
            for ext in ("*.jpg", "*.JPG", "*.jpeg", "*.JPEG", "*.png", "*.PNG"):
                image_files.extend(images_path.glob(ext))
            image_files = sorted(image_files)

            images_processed = 0
            labels_processed = 0

            for img_file in tqdm(image_files, desc=f"  {split}"):
                new_name = f"{prefix}{img_file.name}"

                dst_img = work_dir / split / "images" / new_name
                if not dst_img.exists():
                    try:
                        dst_img.symlink_to(img_file.resolve())
                    except OSError:
                        shutil.copy2(img_file, dst_img)
                    images_processed += 1

                if labels_path and labels_path.exists():
                    txt_label = labels_path / f"{img_file.stem}.txt"
                    if txt_label.exists():
                        dst_label = work_dir / split / "labels" / f"{prefix}{img_file.stem}.txt"
                        if not dst_label.exists():
                            try:
                                dst_label.symlink_to(txt_label.resolve())
                            except OSError:
                                shutil.copy2(txt_label, dst_label)
                            labels_processed += 1

            stats[split]["images"] += images_processed
            stats[split]["labels"] += labels_processed
            print(f"    → {images_processed} images, {labels_processed} labels")

    yaml_config = {
        "path": str(work_dir.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": 1,
        "names": ["pole"],
    }

    yaml_path = work_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_config, f, default_flow_style=False, sort_keys=False)

    print(f"\n{'='*80}")
    print("DATASET PREPARED")
    print(f"{'='*80}")
    for split in ["train", "val", "test"]:
        img_count = len(list((work_dir / split / "images").glob("*")))
        lbl_count = len(list((work_dir / split / "labels").glob("*.txt")))
        print(f"  {split}: {img_count} images, {lbl_count} labels")
    print(f"\n✓ YAML config: {yaml_path}")
    print(f"{'='*80}")

    return yaml_path, stats


def prepare_stage_datasets(stage_definitions, datasets_config, root_work_dir):
    stage_artifacts = {}
    for stage_name, dataset_names in stage_definitions.items():
        stage_dir = root_work_dir / stage_name
        yaml_path, stats = prepare_dataset(
            datasets_config=datasets_config,
            work_dir=stage_dir,
            dataset_names=dataset_names,
        )
        stage_artifacts[stage_name] = {
            "work_dir": stage_dir,
            "data_yaml": yaml_path,
            "stats": stats,
            "datasets": dataset_names,
        }
    return stage_artifacts


In [6]:
# Prepare all stage datasets
STAGE_ARTIFACTS = prepare_stage_datasets(STAGE_DATASETS, DATASETS, WORK_DIR)

print("\n✓ Stage datasets ready!")
for stage_name, artifact in STAGE_ARTIFACTS.items():
    print(f"  {stage_name}: {artifact['data_yaml']}")



Preparing Dataset for Training

Processing roadpoles_v1...


  train: 100%|██████████| 322/322 [00:00<00:00, 644.08it/s]


    → 322 images, 322 labels


  val: 100%|██████████| 92/92 [00:00<00:00, 661.24it/s]


    → 92 images, 92 labels


...

    → 942 images, 942 labels


  val: 100%|██████████| 261/261 [00:00<00:00, 644.54it/s]


    → 261 images, 261 labels


  test: 100%|██████████| 138/138 [00:00<00:00, 1129.16it/s]

    → 138 images, 0 labels

DATASET PREPARED
  train: 1264 images, 1264 labels
  val: 353 images, 353 labels
  test: 184 images, 0 labels

✓ YAML config: training_data/data.yaml

✓ Ready for training!
  Data YAML: training_data/data.yaml


## 4. Training Configuration

Settings below reflect the report:

- YOLO models: **YOLOv9t** and **YOLO11n**
- Thin objects -> higher native resolution (**1280** by default)
- Two-stage training:
  - Stage 1 pre-training on pseudo labels
  - Stage 2 fine-tuning on real domain data
- RF-DETR added as a transformer baseline / ensemble member


In [9]:
# Auto-detect device
if torch.cuda.is_available():
    DEVICE = 0
    GPU_NAME = torch.cuda.get_device_name(0)
    BATCH_SIZE_YOLO = 8 if "4090" in GPU_NAME or "A100" in GPU_NAME else 4
    print("✓ GPU detected!")
    print(f"  GPU: {GPU_NAME}")
else:
    DEVICE = "cpu"
    GPU_NAME = "CPU"
    BATCH_SIZE_YOLO = 2
    print("⚠️  No GPU! Using CPU (very slow)")
    print("  Recommendation: Use IDUN cluster or Cybele lab")

PROJECT_NAME = "snowpole_detection"
YOLO_IMG_SIZE = 1280
YOLO_STAGE1_EPOCHS = 60
YOLO_STAGE2_EPOCHS = 80

YOLO_MODELS = [
    "yolov9t",
    "yolo11n",
]

TRAIN_RF_DETR = True
RFDETR_MODEL_NAME = "medium"   # nano | small | medium | large
RFDETR_STAGE1_EPOCHS = 40
RFDETR_STAGE2_EPOCHS = 60
RFDETR_BATCH_SIZE = 4 if torch.cuda.is_available() else 1
RFDETR_GRAD_ACCUM = 4 if RFDETR_BATCH_SIZE <= 4 else 1
RFDETR_LR = 1e-4

# If Stage 1 dataset is unavailable, training automatically starts from Stage 2
STAGE_SEQUENCE = ["stage1", "stage2"]

print(f"\nTraining Configuration:")
print(f"  Project: {PROJECT_NAME}")
print(f"  YOLO image size: {YOLO_IMG_SIZE}")
print(f"  YOLO batch size: {BATCH_SIZE_YOLO}")
print(f"  YOLO models: {YOLO_MODELS}")
print(f"  RF-DETR enabled: {TRAIN_RF_DETR}")
print(f"  RF-DETR size: {RFDETR_MODEL_NAME}")
print(f"  Device: {DEVICE}")
print(f"  Stages: {STAGE_SEQUENCE}")


✓ GPU detected!
  GPU: NVIDIA GeForce RTX 4090

Training Configuration:
  Project: snowpole_detection
  Image size: 640
  Batch size: 16
  Epochs: 150
  Device: 0
  Models: ['yolov8n', 'yolov9t', 'yolo11n']


In [10]:
def get_yolo_training_config(data_yaml, model_name, img_size, batch_size, epochs, device, stage_name):
    """
    YOLO training config aligned with the project report:
    - High resolution for thin snow poles
    - Strong photometric augmentation for winter weather diversity
    - No rotation because poles remain vertical
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_name = f"{model_name}_{stage_name}_{timestamp}"

    config = {
        "data": str(data_yaml),
        "imgsz": img_size,
        "batch": batch_size,
        "epochs": epochs,
        "device": device,
        "project": f"runs/{PROJECT_NAME}",
        "name": experiment_name,
        "exist_ok": False,

        # Optimization
        "optimizer": "AdamW",
        "lr0": 0.001 if stage_name == "stage1" else 0.0005,
        "lrf": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_epochs": 3.0,

        # Loss weights
        "box": 8.0,
        "cls": 0.3,
        "dfl": 1.5,

        # Weather-aware augmentation
        "hsv_h": 0.01,
        "hsv_s": 0.6,
        "hsv_v": 0.5,

        # Pole-aware geometry
        "degrees": 0.0,
        "translate": 0.10,
        "scale": 0.50,
        "shear": 0.0,
        "perspective": 0.0,
        "flipud": 0.0,
        "fliplr": 0.5,

        # Advanced augmentation
        "mosaic": 1.0 if stage_name == "stage1" else 0.5,
        "mixup": 0.10 if stage_name == "stage1" else 0.05,
        "copy_paste": 0.10,
        "close_mosaic": 10,

        # Training settings
        "cos_lr": True,
        "amp": True,
        "patience": 30,
        "save": True,
        "save_period": 10,
        "cache": False,
        "workers": 8,
        "plots": True,
        "val": True,
        "verbose": True,
    }
    return config, experiment_name


def get_rfdetr_model(size="medium"):
    if not RFDETR_AVAILABLE:
        raise RuntimeError("RF-DETR is not available. Install `rfdetr` first.")

    size = size.lower()
    model_map = {
        "nano": RFDETRNano,
        "small": RFDETRSmall,
        "medium": RFDETRMedium,
        "large": RFDETRLarge,
    }
    if size not in model_map:
        raise ValueError(f"Unknown RF-DETR size: {size}")
    return model_map[size]()


def run_rfdetr_stage(stage_name, dataset_dir, epochs, batch_size, grad_accum_steps, lr, output_dir, resume_from=None):
    """
    Train RF-DETR on a stage dataset.
    RF-DETR supports YOLO datasets through the dataset_dir + data.yaml structure.
    """
    model = get_rfdetr_model(RFDETR_MODEL_NAME)

    kwargs = dict(
        dataset_dir=str(dataset_dir),
        epochs=epochs,
        batch_size=batch_size,
        grad_accum_steps=grad_accum_steps,
        lr=lr,
        output_dir=str(output_dir),
    )

    # Only pass resume / checkpoint path if available and supported by the installed version.
    if resume_from is not None:
        kwargs["resume_from_checkpoint"] = str(resume_from)

    print(f"\nRF-DETR {stage_name} config:")
    pprint(kwargs)

    return model.train(**kwargs)


print("✓ Training configs ready")
print("\nStrategy summary:")
print("  ✓ Two-stage transfer learning")
print("  ✓ YOLOv9t + YOLO11n for CNN detectors")
print("  ✓ RF-DETR for transformer detector")
print("  ✓ High-resolution training for thin poles")


✓ Training config ready

Augmentation strategy:
  ✓ Strong brightness (hsv_v=0.8)
  ✓ High scale variation (0.9)
  ✓ Mosaic + MixUp + Copy-Paste
  ✓ No rotation (poles stay vertical)


## 5. Train Models

This section runs:

1. **YOLOv9t** and **YOLO11n**
   - optional Stage 1 pre-training on pseudo labels
   - Stage 2 fine-tuning on real domain data

2. **RF-DETR**
   - optional Stage 1 pre-training
   - Stage 2 fine-tuning

If `youtube_pseudo` is not configured, Stage 1 is skipped automatically.


In [11]:
results_summary = []
training_start_time = time.time()

def stage_exists_and_nonempty(stage_name):
    artifact = STAGE_ARTIFACTS.get(stage_name)
    if artifact is None:
        return False
    train_dir = artifact["work_dir"] / "train" / "images"
    return train_dir.exists() and len(list(train_dir.glob("*"))) > 0

# -----------------------------
# 1) YOLO two-stage training
# -----------------------------
for model_name in YOLO_MODELS:
    print(f"\n{'='*80}")
    print(f"YOLO PIPELINE: {model_name}")
    print(f"{'='*80}")

    current_weights = f"{model_name}.pt"

    for stage_name in STAGE_SEQUENCE:
        if not stage_exists_and_nonempty(stage_name):
            print(f"\n⚠️ Skipping {stage_name} for {model_name}: dataset not available.")
            continue

        artifact = STAGE_ARTIFACTS[stage_name]
        epochs = YOLO_STAGE1_EPOCHS if stage_name == "stage1" else YOLO_STAGE2_EPOCHS

        try:
            config, experiment_name = get_yolo_training_config(
                data_yaml=artifact["data_yaml"],
                model_name=model_name,
                img_size=YOLO_IMG_SIZE,
                batch_size=BATCH_SIZE_YOLO,
                epochs=epochs,
                device=DEVICE,
                stage_name=stage_name,
            )

            print(f"\nExperiment: {experiment_name}")
            print(f"Output: runs/{PROJECT_NAME}/{experiment_name}")
            print(f"Loading weights: {current_weights}")

            model = YOLO(current_weights)
            start_time = time.time()
            results = model.train(**config)
            training_time = time.time() - start_time

            exp_dir = Path(f"runs/{PROJECT_NAME}/{experiment_name}")
            best_weights = exp_dir / "weights" / "best.pt"
            last_weights = exp_dir / "weights" / "last.pt"

            if best_weights.exists():
                current_weights = str(best_weights)

            train_imgs = len(list((artifact["work_dir"] / "train" / "images").glob("*")))
            val_imgs = len(list((artifact["work_dir"] / "val" / "images").glob("*")))

            metadata = {
                "framework": "ultralytics",
                "model": model_name,
                "stage": stage_name,
                "experiment": experiment_name,
                "datasets": artifact["datasets"],
                "train_images": train_imgs,
                "val_images": val_imgs,
                "epochs": epochs,
                "img_size": YOLO_IMG_SIZE,
                "batch_size": BATCH_SIZE_YOLO,
                "device": str(DEVICE),
                "training_time_seconds": training_time,
                "training_time_hours": training_time / 3600,
                "best_weights": str(best_weights),
                "last_weights": str(last_weights),
                "input_weights": str(config.get("model", current_weights)),
                "timestamp": datetime.now().isoformat(),
            }

            metadata_path = exp_dir / "training_metadata.json"
            with open(metadata_path, "w") as f:
                json.dump(metadata, f, indent=2)

            results_summary.append(metadata)

            print(f"\n✓ {model_name} {stage_name} complete")
            print(f"  Time: {training_time/3600:.2f} hours")
            print(f"  Best weights: {best_weights}")
            print(f"  Metadata: {metadata_path}")

        except Exception as e:
            print(f"\n✗ Error training {model_name} on {stage_name}: {e}")
            import traceback
            traceback.print_exc()

# -----------------------------
# 2) RF-DETR two-stage training
# -----------------------------
if TRAIN_RF_DETR:
    print(f"\n{'='*80}")
    print(f"RF-DETR PIPELINE: {RFDETR_MODEL_NAME}")
    print(f"{'='*80}")

    previous_stage_checkpoint = None

    for stage_name in STAGE_SEQUENCE:
        if not stage_exists_and_nonempty(stage_name):
            print(f"\n⚠️ Skipping RF-DETR {stage_name}: dataset not available.")
            continue

        artifact = STAGE_ARTIFACTS[stage_name]
        epochs = RFDETR_STAGE1_EPOCHS if stage_name == "stage1" else RFDETR_STAGE2_EPOCHS
        rf_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = RUNS_DIR / PROJECT_NAME / f"rfdetr_{RFDETR_MODEL_NAME}_{stage_name}_{rf_timestamp}"

        try:
            start_time = time.time()
            run_rfdetr_stage(
                stage_name=stage_name,
                dataset_dir=artifact["work_dir"],
                epochs=epochs,
                batch_size=RFDETR_BATCH_SIZE,
                grad_accum_steps=RFDETR_GRAD_ACCUM,
                lr=RFDETR_LR if stage_name == "stage1" else RFDETR_LR / 2,
                output_dir=output_dir,
                resume_from=previous_stage_checkpoint,
            )
            training_time = time.time() - start_time

            # Common checkpoint names to search for after training
            candidate_checkpoints = [
                output_dir / "checkpoint_best_regular.pth",
                output_dir / "checkpoint_best.pth",
                output_dir / "checkpoint.pth",
                output_dir / "best.pth",
            ]
            best_checkpoint = next((p for p in candidate_checkpoints if p.exists()), None)
            if best_checkpoint is not None:
                previous_stage_checkpoint = best_checkpoint

            train_imgs = len(list((artifact["work_dir"] / "train" / "images").glob("*")))
            val_imgs = len(list((artifact["work_dir"] / "val" / "images").glob("*")))

            metadata = {
                "framework": "rfdetr",
                "model": f"rfdetr_{RFDETR_MODEL_NAME}",
                "stage": stage_name,
                "experiment": output_dir.name,
                "datasets": artifact["datasets"],
                "train_images": train_imgs,
                "val_images": val_imgs,
                "epochs": epochs,
                "batch_size": RFDETR_BATCH_SIZE,
                "grad_accum_steps": RFDETR_GRAD_ACCUM,
                "learning_rate": RFDETR_LR if stage_name == "stage1" else RFDETR_LR / 2,
                "device": str(DEVICE),
                "training_time_seconds": training_time,
                "training_time_hours": training_time / 3600,
                "best_weights": str(best_checkpoint) if best_checkpoint else "",
                "timestamp": datetime.now().isoformat(),
            }

            output_dir.mkdir(parents=True, exist_ok=True)
            metadata_path = output_dir / "training_metadata.json"
            with open(metadata_path, "w") as f:
                json.dump(metadata, f, indent=2)

            results_summary.append(metadata)

            print(f"\n✓ RF-DETR {stage_name} complete")
            print(f"  Time: {training_time/3600:.2f} hours")
            print(f"  Best checkpoint: {best_checkpoint}")
            print(f"  Metadata: {metadata_path}")

        except TypeError as e:
            print(f"\n✗ RF-DETR API mismatch on {stage_name}: {e}")
            print("  Check the installed rfdetr version and remove/rename unsupported arguments.")
            import traceback
            traceback.print_exc()
        except Exception as e:
            print(f"\n✗ Error training RF-DETR on {stage_name}: {e}")
            import traceback
            traceback.print_exc()

total_time = time.time() - training_start_time
print(f"\n{'='*80}")
print("All Training Complete!")
print(f"Total time: {total_time/3600:.2f} hours")
print(f"{'='*80}")



Training Model: yolov8n

Experiment: yolov8n_20251122_112324
Output: runs/snowpole_detection/yolov8n_20251122_112324

Loading pretrained: yolov8n.pt
✓ Model loaded

Starting training...

New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24058MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=training_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.7, hsv_v=0.8, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, 

## 6. Training Summary

In [12]:
if results_summary:
    df_summary = pd.DataFrame(results_summary)
    display_cols = [
        c for c in [
            "framework", "model", "stage", "datasets", "train_images", "val_images",
            "epochs", "img_size", "batch_size", "grad_accum_steps",
            "training_time_hours", "best_weights"
        ] if c in df_summary.columns
    ]

    print(f"\n{'='*80}")
    print("TRAINING SUMMARY")
    print(f"{'='*80}")
    print(df_summary[display_cols].to_string(index=False))
else:
    print("No training runs were completed.")



TRAINING SUMMARY
  Model  Dataset  Train Images  Val Images  Epochs Time (hours) Best Weights
yolov8n combined          1264         353     150         0.17      best.pt
yolov9t combined          1264         353     150         0.29      best.pt
yolo11n combined          1264         353     150         0.18      best.pt

✓ Summary: runs/snowpole_detection/training_summary.json
✓ CSV: runs/snowpole_detection/training_summary.csv


## 7. Visualize Training Results

In [13]:
if results_summary:
    for result in results_summary:
        exp_dir = Path(f"runs/{PROJECT_NAME}/{result['experiment']}")
        results_csv = exp_dir / 'results.csv'
        
        if results_csv.exists():
            print(f"\n{'='*80}")
            print(f"Model: {result['model']}")
            print(f"{'='*80}")
            
            df = pd.read_csv(results_csv)
            df.columns = df.columns.str.strip()
            
            # Final metrics
            final = df.iloc[-1]
            print(f"\nFinal Metrics (Epoch {int(final['epoch'])}):") 
            
            for metric in ['metrics/precision(B)', 'metrics/recall(B)', 
                          'metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
                if metric in df.columns:
                    print(f"  {metric}: {final[metric]:.4f}")
            
            # Plot
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            
            if 'train/box_loss' in df.columns:
                axes[0,0].plot(df['epoch'], df['train/box_loss'], label='Train')
                if 'val/box_loss' in df.columns:
                    axes[0,0].plot(df['epoch'], df['val/box_loss'], label='Val')
                axes[0,0].set_title('Box Loss')
                axes[0,0].legend()
                axes[0,0].grid(True)
            
            if 'metrics/mAP50(B)' in df.columns:
                axes[0,1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
                axes[0,1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
                axes[0,1].set_title('mAP')
                axes[0,1].legend()
                axes[0,1].grid(True)
            
            if 'metrics/precision(B)' in df.columns:
                axes[1,0].plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
                axes[1,0].plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
                axes[1,0].set_title('Precision & Recall')
                axes[1,0].legend()
                axes[1,0].grid(True)
            
            if 'lr/pg0' in df.columns:
                axes[1,1].plot(df['epoch'], df['lr/pg0'])
                axes[1,1].set_title('Learning Rate')
                axes[1,1].grid(True)
            
            plt.suptitle(f"{result['model']} Training", fontweight='bold')
            plt.tight_layout()
            
            plot_path = exp_dir / 'training_curves.png'
            plt.savefig(plot_path, dpi=150)
            plt.show()
            print(f"\n✓ Plot saved: {plot_path}")
else:
    print("No results to visualize")


Model: yolov8n

Final Metrics (Epoch 150):
  metrics/precision(B): 0.9105
  metrics/recall(B): 0.7765
  metrics/mAP50(B): 0.8827
  metrics/mAP50-95(B): 0.6794


<Figure size 1200x1000 with 4 Axes>


✓ Plot saved: runs/snowpole_detection/yolov8n_20251122_112324/training_curves.png

Model: yolov9t

Final Metrics (Epoch 150):
  metrics/precision(B): 0.9057
  metrics/recall(B): 0.7675
  metrics/mAP50(B): 0.8675
  metrics/mAP50-95(B): 0.6628


<Figure size 1200x1000 with 4 Axes>


✓ Plot saved: runs/snowpole_detection/yolov9t_20251122_113325/training_curves.png

Model: yolo11n

Final Metrics (Epoch 150):
  metrics/precision(B): 0.9207
  metrics/recall(B): 0.8386
  metrics/mAP50(B): 0.9110
  metrics/mAP50-95(B): 0.6795


<Figure size 1200x1000 with 4 Axes>


✓ Plot saved: runs/snowpole_detection/yolo11n_20251122_115057/training_curves.png


## 8. Sustainability Analysis

In [14]:
if results_summary:
    print(f"\n{'='*80}")
    print("SUSTAINABILITY ANALYSIS")
    print(f"{'='*80}")
    
    # GPU power estimates (typical values)
    GPU_POWER = {
        'RTX 4090': 450,      # Watts
        'RTX 3090': 350,
        'RTX 3080': 320,
        'A100': 400,
        'V100': 300,
        'CPU': 150,
    }
    
    # Tesla Model Y efficiency: ~150 Wh/km
    TESLA_WH_PER_KM = 150
    TRONDHEIM_TO_OSLO_KM = 500
    
    for result in results_summary:
        hours = result['training_time_hours']
        
        # Estimate power (use RTX 4090 for Cybele, or CPU)
        if result['device'] == 'cpu':
            power_watts = GPU_POWER['CPU']
            device_name = 'CPU'
        else:
            power_watts = GPU_POWER['RTX 4090']  # Cybele lab
            device_name = 'RTX 4090'
        
        # Energy calculation
        energy_kwh = (power_watts * hours) / 1000
        energy_wh = energy_kwh * 1000
        
        # Tesla distance
        tesla_km = energy_wh / TESLA_WH_PER_KM
        tesla_pct = (tesla_km / TRONDHEIM_TO_OSLO_KM) * 100
        
        print(f"\n{result['model']}:")
        print(f"  Training time: {hours:.2f} hours")
        print(f"  Device: {device_name} ({power_watts}W)")
        print(f"  Energy consumption: {energy_kwh:.2f} kWh")
        print(f"  Tesla Model Y equivalent: {tesla_km:.1f} km ({tesla_pct:.1f}% of Trondheim→Oslo)")
    
    print(f"\n{'='*80}")
else:
    print("No training data for sustainability analysis")


SUSTAINABILITY ANALYSIS

yolov8n:
  Training time: 0.17 hours
  Device: RTX 4090 (450W)
  Energy consumption: 0.08 kWh
  Tesla Model Y equivalent: 0.5 km (0.1% of Trondheim→Oslo)

yolov9t:
  Training time: 0.29 hours
  Device: RTX 4090 (450W)
  Energy consumption: 0.13 kWh
  Tesla Model Y equivalent: 0.9 km (0.2% of Trondheim→Oslo)

yolo11n:
  Training time: 0.18 hours
  Device: RTX 4090 (450W)
  Energy consumption: 0.08 kWh
  Tesla Model Y equivalent: 0.5 km (0.1% of Trondheim→Oslo)



## 9. Next Steps

In [15]:
print(f"\n{'='*80}")
print("NEXT STEPS")
print(f"{'='*80}")
print("\n1. Evaluate models:")
print("   - Run Evaluation.ipynb with the best YOLO and RF-DETR checkpoints")
print("   - Compare mAP@50 and mAP@50:95")
print("\n2. Reproduce the reported strategy:")
print("   - Stage 1: pseudo-labeled YouTube data")
print("   - Stage 2: fine-tune on iPhone / Roadpoles")
print("   - Use TTA + WBF in inference / ensemble notebook")
print("\n3. High-resolution experiments:")
print("   - Try YOLO_IMG_SIZE = 1920 for distant 1–2 pixel poles")
print("   - Reduce batch size accordingly")
print("\n4. Ensemble:")
print("   - Combine YOLOv9t + YOLO11n + RF-DETR with WBF")
print("   - Measure accuracy vs latency on target hardware")
print(f"{'='*80}")

if results_summary:
    summary_path = RUNS_DIR / PROJECT_NAME / 'all_training_runs_summary.json'
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(summary_path, 'w') as f:
        json.dump(results_summary, f, indent=2)
    print(f"\n✓ Full summary saved to: {summary_path}")



NEXT STEPS

1. Evaluate models:
   - Run Evaluation.ipynb with trained weights
   - Calculate Precision, Recall, mAP@50, mAP@0.5:0.95

2. Check outputs:
   - Results: runs/snowpole_detection/
   - Training curves, confusion matrices, PR curves

3. Model comparison:
   - Compare different YOLO variants
   - Accuracy vs Speed tradeoff

📁 Trained models:
   yolov8n: runs/snowpole_detection/yolov8n_20251122_112324/weights/best.pt
   yolov9t: runs/snowpole_detection/yolov9t_20251122_113325/weights/best.pt
   yolo11n: runs/snowpole_detection/yolo11n_20251122_115057/weights/best.pt
